# Zerith local RAG chatbot: operations and experiments

This notebook is a debug client for the separate FastAPI service. The always-on runtime is `app.main` (Uvicorn), not the notebook kernel. Ollama serves the local LLM and embedding model; Qdrant persists and searches document vectors.

## Start the local dependencies

1. Install/start Ollama, then run `ollama pull qwen2.5:3b` and `ollama pull nomic-embed-text`.
2. Start persistent Qdrant with `docker compose up -d qdrant` from `chatbot-service`, or configure `QDRANT_URL` and optional `QDRANT_API_KEY` for a managed instance.
3. Create `.venv`, install `requirements.txt`, then run `run_chatbot_service.bat` in a separate terminal.
4. Put learning notes in `rag-corpus/`. Supported files: Markdown, text, and JSON Lines. Do not index secrets or student data.
5. Run cells in order to check readiness, index the corpus, inspect retrieval results, and test an answer. Set `CHATBOT_API_TOKEN` and `CHATBOT_ADMIN_TOKEN` as notebook environment variables if configured on the service.

The default chat model is small for local development and is configurable. On limited VRAM, change the model or set `OLLAMA_KEEP_ALIVE=30m` to release it after inactivity.

In [ ]:
import json
import os
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

CHATBOT_URL = os.getenv("CHATBOT_SERVICE_URL", "http://127.0.0.1:8100").rstrip("/")
SERVICE_DIR = Path.cwd()
if SERVICE_DIR.name != "chatbot-service":
    SERVICE_DIR = SERVICE_DIR / "chatbot-service"
CORPUS_DIR = Path(os.getenv("CHATBOT_CORPUS_DIR", str(SERVICE_DIR / "rag-corpus")))
API_TOKEN = os.getenv("CHATBOT_API_TOKEN", "")
ADMIN_TOKEN = os.getenv("CHATBOT_ADMIN_TOKEN", "")

def call_service(path, payload=None, admin=False):
    headers = {"Accept": "application/json"}
    if payload is not None:
        headers["Content-Type"] = "application/json"
    if API_TOKEN:
        headers["X-Chatbot-Api-Token"] = API_TOKEN
    if admin and ADMIN_TOKEN:
        headers["X-Chatbot-Admin-Token"] = ADMIN_TOKEN
    body = json.dumps(payload).encode("utf-8") if payload is not None else None
    request = Request(CHATBOT_URL + path, data=body, headers=headers, method="POST" if body else "GET")
    try:
        with urlopen(request, timeout=180) as response:
            return json.loads(response.read().decode("utf-8"))
    except HTTPError as error:
        detail = error.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Service returned HTTP {error.code}: {detail}") from error
    except URLError as error:
        raise RuntimeError(f"Cannot reach chatbot service at {CHATBOT_URL}: {error.reason}") from error

print("Service:", CHATBOT_URL)
print("Corpus:", CORPUS_DIR.resolve())

In [ ]:
# Basic process health is distinct from model/vector-store readiness.
print(call_service("/health"))
print(json.dumps(call_service("/ready"), indent=2))

In [ ]:
# Index (or refresh) the corpus into persistent Qdrant storage.
corpus_files = sorted(
    path.relative_to(CORPUS_DIR).as_posix()
    for path in CORPUS_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in {".md", ".txt", ".jsonl"}
)
if not corpus_files:
    raise FileNotFoundError(f"No supported documents found under {CORPUS_DIR.resolve()}")
print("Files to index:", corpus_files)
print(call_service("/admin/index", payload={}, admin=True))

In [ ]:
# Inspect retrieval without generating an answer. Adjust query/topK to diagnose grounding.
query = "How can I reason about array boundaries and an empty input?"
retrieval = call_service("/debug/retrieve", payload={"query": query, "topK": 4})
for result in retrieval["data"]["results"]:
    print(f"{result['score']:.3f} | {result['source']} | chunk {result['chunk_index']}")
    print(result["text"], "\n")

In [ ]:
# Exercise the same local service contract used by the Express chatbot.
system_prompt = "You are ZAI, a warm Socratic DSA coding mentor. Never give code or the full solution. Ask one concise guiding question."
chat_result = call_service("/chat", payload={
    "systemPrompt": system_prompt,
    "messages": [{"role": "user", "content": "How should I start checking this array algorithm for edge cases?"}],
    "topK": 4,
})
print("Answer:\n", chat_result["data"]["reply"])
print("\nRetrieved sources:")
for source in chat_result["data"]["sources"]:
    print(source)